# Search 50 FAQs by meaning

**dabeer.ai · AI for Beginners** · Chat with your documents

In this notebook you'll search the help pages of **Nakshatra Electronics**, a fictional online electronics shop, in two ways:

1. **Keyword search**: finds FAQs that share words with your question.
2. **Meaning search**: turns every FAQ and your question into embeddings, and finds the closest ones.

Everything runs inside Colab with a free, open embedding model. **No API key is needed.**

Run the cells from top to bottom with **Shift + Enter**.

## Step 1: Install and set up

Colab usually has this library already. The cell below makes sure. It takes about a minute the first time.

In [ ]:
%pip install -q sentence-transformers pandas

## Step 2: Settings

Everything you might want to change is in this one cell.

In [ ]:
CSV_PATH = "shop-faqs.csv"     # the starter file from this section's starter/ folder
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"   # small, free, runs on a normal CPU
TOP_K = 3                      # how many results to show for each search

## Step 3: Load the 50 FAQs

If the file isn't in Colab yet, this cell opens an upload button. Choose `shop-faqs.csv`.

In [ ]:
import os
import pandas as pd

if not os.path.exists(CSV_PATH):
    from google.colab import files   # only works inside Colab
    uploaded = files.upload()
    CSV_PATH = next(iter(uploaded))

faqs = pd.read_csv(CSV_PATH)
print(len(faqs), "FAQs loaded")
faqs.head(10)

## Step 4: Keyword search

This is the old way. It removes common words like "the" and "my", then counts how many of your words appear in each FAQ (question and answer).

In [ ]:
import re

STOP_WORDS = set("""a an the and or but if of to in on at for from by with is are was were be been
i me my we our you your it its this that these those do does did can could will would should
how what when where why which who whom not no yes so than then there here have has had get got""".split())

def words(text):
    return {w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP_WORDS}

def keyword_search(query, k=TOP_K):
    q = words(query)
    scores = []
    for i, row in faqs.iterrows():
        overlap = q & words(row["question"] + " " + row["answer"])
        scores.append((len(overlap), i, sorted(overlap)))
    scores.sort(key=lambda s: (-s[0], s[1]))
    print("Keyword search:", query)
    for n, i, matched in scores[:k]:
        if n == 0:
            print("   (no more FAQs share a word with your question)")
            break
        print(f"   {n} shared words {matched} -> FAQ {faqs.loc[i, 'id']}: {faqs.loc[i, 'question']}")

keyword_search("What is your return policy?")
keyword_search("When do I get my money back?")

## Step 5: Turn every FAQ into an embedding

The model reads each FAQ and returns a list of 384 numbers: its position on the "map of meaning". We embed the question and answer together.

The first run downloads the model (about 90 MB).

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMBED_MODEL)

faq_texts = (faqs["question"] + " " + faqs["answer"]).tolist()
faq_vectors = model.encode(faq_texts, normalize_embeddings=True, show_progress_bar=True)

print("Shape:", faq_vectors.shape, "(FAQs, numbers per FAQ)")
print("The first 8 numbers of FAQ 1:", faq_vectors[0][:8].round(3))

## Step 6: Cosine similarity, by hand once

Cosine similarity measures how closely two embeddings point the same way:

- close to **1**: very similar meaning
- around **0**: unrelated

Because we asked for `normalize_embeddings=True`, cosine similarity is just the dot product of two vectors. Try the three sentences from the lesson.

In [ ]:
import numpy as np

def similarity(a, b):
    va, vb = model.encode([a, b], normalize_embeddings=True)
    return float(np.dot(va, vb))

pairs = [
    ("Cancel my subscription", "Stop my monthly plan"),
    ("Cancel my subscription", "Subscribe to the newsletter"),
    ("refund status, please", "when do I get my money back"),
    ("refund status, please", "which TV size suits my room"),
]
for a, b in pairs:
    print(f"{similarity(a, b):.2f}  '{a}'  vs  '{b}'")

## Step 7: Meaning search

Embed the question with the **same model**, then rank all 50 FAQs by similarity.

In [ ]:
def meaning_search(query, k=TOP_K):
    q_vec = model.encode([query], normalize_embeddings=True)[0]
    scores = faq_vectors @ q_vec            # one similarity score per FAQ
    best = np.argsort(-scores)[:k]
    print("Meaning search:", query)
    for i in best:
        print(f"   {scores[i]:.2f} -> FAQ {faqs.loc[i, 'id']}: {faqs.loc[i, 'question']}")
    return [(int(faqs.loc[i, "id"]), float(scores[i])) for i in best]

meaning_search("When do I get my money back?");

## Step 8: Keyword vs meaning, side by side

Run the same questions through both searches. For each one, decide which search found the right FAQ.

In [ ]:
TEST_QUESTIONS = [
    "When do I get my money back?",
    "My UPI payment went through, but the order failed",
    "Can someone come and set up my new AC?",
    "I forgot my login details",
    "Is it possible to pay in monthly instalments without interest?",
    "The box was crushed when it arrived",
]

for q in TEST_QUESTIONS:
    keyword_search(q, k=1)
    meaning_search(q, k=1)
    print()

## Step 9: Your turn

Type your own questions. Try:

- a badly worded question, with spelling mistakes
- a question in Hinglish, like `paisa wapas kab milega`
- a question the shop has **no** FAQ for, like `Do you repair bicycles?`

Look at the scores. What score does a good match get? What score does an unrelated question get?

In [ ]:
my_question = "do u deliver same day in pune"
keyword_search(my_question)
meaning_search(my_question);

## Step 10: Don't answer when nothing matches

A good FAQ bot says "I'm not sure" instead of showing a random answer. Pick a threshold from what you saw in Step 9.

In [ ]:
THRESHOLD = 0.45   # change this after looking at your own scores

def answer(query):
    q_vec = model.encode([query], normalize_embeddings=True)[0]
    scores = faq_vectors @ q_vec
    i = int(np.argmax(scores))
    if scores[i] < THRESHOLD:
        return f"Sorry, I couldn't find an answer to that. (best score {scores[i]:.2f})"
    return f"{faqs.loc[i, 'answer']}  (FAQ {faqs.loc[i, 'id']}, score {scores[i]:.2f})"

for q in ["How do I return a phone?", "Do you repair bicycles?", "What is the capital of France?"]:
    print(q, "\n   ", answer(q), "\n")